No Databricks

https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/853238798589295?o=7474657872577658#command/5894851456878343

# Demo: Model Tuning com Ray Tune

Nesta demonstração, aprenderá a utilizar o **Ray Tune** (uma estrutura poderosa de otimização de hiperparâmetros) para afinar modelos de machine learning no Databricks.

Demonstraremos como implementar a biblioteca Ray Tune usando o **Random Forest Regressor** do Scikit-Learn, cobrindo:
* Definição de espaços de pesquisa (*search spaces*)
* Criação de funções objetivo
* Otimização de hiperparâmetros

Além disso, rastrearemos e registaremos os resultados com o **MLflow**, permitindo uma gestão e monitorização eficientes do processo de afinação.

---

### Objetivos de Aprendizagem

Ao final desta demonstração, será capaz de:
* Definir uma **função objetivo** específica para o Ray Tune.
* Configurar espaços de pesquisa ao estilo **Optuna** dentro do Ray Tune.
* Configurar o **Tuner** do Ray Tune e os respetivos recursos de computação.
* Otimizar hiperparâmetros utilizando **execução paralela**.

## Model Tuning with Ray Tune and a Single-Machine Model

Nesta parte, utilizaremos o **Ray** para otimização distribuída de hiperparâmetros enquanto treinamos um modelo do Scikit-Learn.

### How This Works:

* **Conversão de Dados:** Os dados são convertidos de um Spark DataFrame para Pandas para permitir o treino de modelos numa única máquina.
* **Execução Distribuída:** O Ray Tune é executado numa única máquina, mas distribui a execução dos testes (*trials*) através de *multiple CPU threads* para acelerar a afinação dos hiperparâmetros.
* **Rastreio com MLflow:** O MLflow regista a experiência, guardando os melhores hiperparâmetros e o desempenho do modelo.

Esta abordagem permite utilizar o Ray para uma pesquisa distribuída de hiperparâmetros, mantendo o treino do modelo num único nó para tirar partido das implementações eficientes do Scikit-Learn.

In [0]:
%pip install -U optuna optuna-integration mlflow
%pip install --upgrade ray[tune]
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Before starting the demo, run the provided classroom setup script.

In [0]:
#%run ../Includes/Classroom-Setup-02.1b

### Other Conventions:

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
username = 'fabieneaulas@gmail.com'
catalog_name='workspace'
schema_name= 'default'
print(f"Username:          {username}")
print(f"Catalog Name:      {catalog_name}")
print(f"Schema Name:       {schema_name}")
#print(f"Working Directory: {DA.paths.working_dir}")
#print(f"Dataset Location:  {DA.paths.datasets.wine_quality}")

Username:          fabieneaulas@gmail.com
Catalog Name:      workspace
Schema Name:       default


## Configure a Ray Cluster

Let's begin by setting up a single-machine (driver-only) Ray cluster by defining the following:

* 3 CPU cores allocated for the head node, leaving 1 core for Spark[cite: 5]
  > The Vocareum environment allocates 4 CPUs per user for this demonstration.[cite: 5]
* 1 worker node[cite: 5]
* 4 CPUs per worker[cite: 5]

Additionally, we will initialize the Ray cluster on spark with the above configurations with `ray.init()` and defining the environment variable `RAY_ADDRESS`.[cite: 5]

In [0]:


import os
import ray

# Attempt to shut down any existing Ray instance
try:
    ray.shutdown()
    print("Existing Ray instance shutdown successfully.")
except Exception as e:
    print(f"Warning: No active Ray instance to shut down. Details: {e}")

# Set up configurations for a single-machine (driver-only) Ray cluster
num_cpus_head_node = 3  # Use 3 CPU cores, leaving 1 for Spark

# Initialize Ray locally (setup_ray_cluster requires SparkContext,
# which is not available on serverless compute)
ray.init(ignore_reinit_error=True, num_cpus=num_cpus_head_node, include_dashboard=False)
print("Ray initialized successfully.")

# Set Ray address for local mode
os.environ['RAY_ADDRESS'] = "local"

Existing Ray instance shutdown successfully.


I0000 00:00:1789785621.465103      75 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1789785621.688635      75 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1789785621.909956      75 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
2026-09-19 02:40:22,868	WARNING services.py:2248 -- WARNING: The object store is using /local_disk0/tmp/ray instead of /dev/shm because /dev/shm has only 67108864 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=4.91gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
I0000 00:00:1789785622.871490      75 fork_posix.cc:77] Other threads are currently calling into gRPC, skippin

Ray initialized successfully.


In [0]:
#página 5

In [0]:


#import os
#import ray
#from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

# Attempt to shut down any existing Ray cluster
#try:
#    shutdown_ray_cluster()
#    print("Existing Ray cluster shutdown successfully.")
#except Exception as e:
#    print(f"Warning: No active Ray cluster to shut down. Details: {e}")

# Set up configurations for a single-machine (driver-only) Ray cluster
#num_cpus_head_node = 3  # Use 3 CPU cores, leaving 1 for Spark
#num_worker_nodes = 1    # Single-node setup (driver only)
#num_cpu_cores_per_worker = 4  # Unused since there's only one worker

# Initialize the Ray cluster on Spark
#ray_conf = setup_ray_cluster(
#    min_worker_nodes=num_worker_nodes,
#    max_worker_nodes=num_worker_nodes,
#    num_cpus_head_node=num_cpus_head_node,
#    num_gpus_head_node=0  # No GPU usage
#)

# Initialize Ray with the configured settings
#ray.init(ignore_reinit_error=True)
#print(f"Ray initialized with address: {ray_conf[0]}")

# Set Ray address for Spark integration
#os.environ['RAY_ADDRESS'] = ray_conf[0]

## Data Preparation for Distributed Optuna with Single-Machine Training

In this task, before we define the objective function, we need to convert our Spark DataFrame to a Pandas DataFrame. This will allow us to split the data into training and test sets and standardize the features. These steps are essential for conducting single-node model training using Scikit-learn.

### Instructions:

1. **Convert the Spark DataFrame into a Pandas DataFrame** for single-node processing.
2. **Split the dataset into training and test sets** using Scikit-learn's `train_test_split` method.
3. **Standardize the features** to ensure the model performs optimally.

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the wine quality dataset and define feature/label columns
# (normally provided by the classroom setup script)
train_df = spark.read.table("workspace.default.delta_table_wine06092026")
label_column = "quality"
feature_columns = [c for c in train_df.columns if c != label_column]

# Convert Spark DataFrame to Pandas for single-machine training
train_pandas = train_df.toPandas()

# Separate features and labels
X = train_pandas[feature_columns]
y = train_pandas[label_column]

# Split the data into training and test sets using Scikit-learn
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Display the shapes of the training and test sets
print(f"Training data shape: {X_train.shape}, Test data shape: {X_test.shape}")

Training data shape: (1271, 11), Test data shape: (318, 11)


In [0]:
#página 7

## Define the Ray Tune Objective Function and Search Space for Distributed Hyperparameter Search (Single-Machine Training)

In this step, we will define the **objective function** for hyperparameter tuning using Ray Tune.

This function will:

* **Suggest hyperparameters** dynamically.
* **Train and evaluate** a `RandomForest` model using Scikit-learn.
* **Utilize cross-validation** for performance evaluation.

---

### Key Differences from the Previous Demonstration

* Unlike the previous approach, we now **integrate cross-validation** for model evaluation.
* Cross-validation is feasible in single-node training but can be **resource-intensive** in a distributed setup.

---

### Instructions

1. **Define the Hyperparameter Search Space**
   * Use Ray Tune's API for defining search spaces.
   * [Tune Search Space API](https://docs.ray.io/en/latest/tune/api/search_space.html)

2. **Configure the Search Algorithm**
   * Use Ray Tune's Search Algorithms for efficient exploration.
   * [Tune Search Algorithms](https://docs.ray.io/en/latest/tune/api/suggestion.html)


3. **Implement the Objective Function**
   * Train a `RandomForest` model using Scikit-learn.
   * Optimize hyperparameters within the function.

4. **Set Up & Execute the Tuning Process**
   * Use Ray Tune Execution (`tune.Tuner`) to configure and run the tuning process.
   * Apply key configurations like:
     * `TuneConfig`
     * `RunConfig`
     * `CheckpointConfig`


Tune Search Space API

https://docs.ray.io/en/latest/tune/api/search_space.html




Tune Search Algorithms

https://docs.ray.io/en/latest/tune/key-concepts.html

https://docs.ray.io/en/latest/tune/api/suggestion.html

4. **Set Up & Execute the Tuning Process**
   * Use Ray Tune Execution (`tune.Tuner`) to configure and run the tuning process.
   * Apply key configurations like:
     * `TuneConfig`
     * `RunConfig`
     * `CheckpointConfig`
     * `FailureConfig`
   * [Tune Execution (`tune.Tuner`)](https://docs.ray.io/en/latest/tune/api/execution.html)
   * [ray.tune.with_parameters](https://docs.ray.io/en/latest/tune/api/doc/ray.tune.with_parameters.html)

5. **Evaluate Model Performance**
   * Use cross-validation and return negative RMSE (to be minimized).

In [0]:
#página 9 e 13,14

In [0]:
from ray import tune

# Define the hyperparameter search space for RandomForest tuning
search_space = {
    "n_estimators": tune.randint(50, 300),  # Number of trees in the forest (wider range)
    "max_depth": tune.randint(3, 30)        # Depth of the trees (realistic range)
}

In [0]:
# página 15

In [0]:
import mlflow
from mlflow.types.utils import _infer_schema
from mlflow.exceptions import MlflowException
from mlflow.models.signature import infer_signature
from mlflow.utils.databricks_utils import get_databricks_env_vars
from ray import tune
from ray.air.integrations.mlflow import MLflowLoggerCallback, setup_mlflow
from ray.tune.search import ConcurrencyLimiter
from ray.tune.search.optuna import OptunaSearch

In [0]:
# Retrieve Databricks MLflow credentials
mlflow_db_creds = get_databricks_env_vars("databricks")

if not mlflow_db_creds:
    raise ValueError("Databricks MLflow credentials could not be retrieved.")

# Set up MLflow experiment
# MLflow Experiment Setup
experiment_name_ray = os.path.join(
    os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()),
    "02b - Model Tuning with Ray"
)

mlflow.set_experiment(experiment_name_ray)
experiment_id_ray = mlflow.get_experiment_by_name(experiment_name_ray).experiment_id

# Define Optuna search algorithm
searcher = OptunaSearch(metric="rmse", mode="min")  # Minimize RMSE
algo = ConcurrencyLimiter(searcher, max_concurrent=3)  # Limit concurrent trials to 3

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [0]:
import pandas as pd
import mlflow
import os
import ray
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from typing import Dict, Any


def objective_ray_scikit(
    config: Dict[str, Any],
    parent_run_id: str,
    X_train_in: pd.DataFrame,
    y_train_in: pd.Series,
    experiment_name_in: str,
    mlflow_db_creds_in: Dict[str, str],
):
    """Objective function for Ray Tune hyperparameter optimization using
    cross-validation.

    Args:
        config (Dict[str, Any]): Hyperparameter configuration from Ray Tune.
        parent_run_id (str): MLflow parent run ID for nested tracking.
        X_train_in (pd.DataFrame): Training features.
        y_train_in (pd.Series): Training labels.
        experiment_name_in (str): MLflow experiment name.
        mlflow_db_creds_in (Dict[str, str]): Databricks MLflow credentials.

    Returns:
        Dict[str, float]: Dictionary containing RMSE (lower is better).
    """

    try:
        # Update Databricks credentials for Ray (executors restart each run)
        if mlflow_db_creds_in:
            os.environ.update(mlflow_db_creds_in)

        # Start nested MLflow run under the parent run
        with mlflow.start_run(nested=True, experiment_id=experiment_id_ray, tags={"mlflow.parentRunId": parent_run_id}):
            # Extract hyperparameters from Ray Tune config
            n_estimators = config["n_estimators"]
            max_depth = config["max_depth"]

            # Initialize RandomForest Regressor
            model = RandomForestRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=42
            )

            # Perform 3-fold cross-validation and compute RMSE
            scores = cross_val_score(
                model, X_train_in, y_train_in,
                scoring='neg_root_mean_squared_error',
                cv=3
            )

            mean_rmse = -scores.mean()  # Convert negative RMSE to positive

            # Log hyperparameters and metrics in MLflow
            mlflow.log_params(config)
            mlflow.log_metric("RMSE", mean_rmse)

            return {"rmse": mean_rmse}

    except Exception as e:
        print(f"Error in objective function: {e}")
        return {"rmse": float("inf")}  # Return a large RMSE in case of failure            

In [0]:
# página 17/18, slide 17

In [0]:
# Start the parent MLflow run
with mlflow.start_run(run_name="ray_tune", experiment_id=experiment_id_ray) as parent_run:
    os.environ.update(mlflow_db_creds)  # Ensure Ray executors have credentials[cite: 15]

    # Pass search space to OptunaSearch (avoids param_space conflict with Tuner)
    searcher = OptunaSearch(metric="rmse", mode="min", space=search_space)
    algo = ConcurrencyLimiter(searcher, max_concurrent=3)

    # Set up and execute Ray Tune with the objective function and search space[cite: 15]
    tuner = tune.Tuner(
        tune.with_parameters(
            objective_ray_scikit,
            parent_run_id=parent_run.info.run_id,
            X_train_in=X_train,
            y_train_in=y_train,
            experiment_name_in=experiment_name_ray,
            mlflow_db_creds_in=mlflow_db_creds
        ),
        tune_config=tune.TuneConfig(
            search_alg=algo,
            num_samples=10,
            reuse_actors=True  # Keeps actors alive for efficiency[cite: 15]
        )
    )


    # Run tuning and retrieve the best result
    multinode_results = tuner.fit()
    best_result = multinode_results.get_best_result(metric="rmse", mode="min", scope="last")

    if best_result is None:
        raise ValueError("No best trial found. Ensure the tuning job ran successfully.")

    # Extract best trial details
    best_trial_number = best_result.metrics.get("trial_id", "N/A")  # Default if missing
    best_model_params = best_result.config
    best_model_params["random_state"] = 42  # Ensures reproducibility
    best_rmse = best_result.metrics["rmse"]

    # Train the best model using the best hyperparameters
    # Train the best model using the best hyperparameters
    best_model = RandomForestRegressor(**best_model_params)
    best_model.fit(X_train, y_train)

    # Enable MLflow autologging (disable model logging to avoid conflicts)
    mlflow.sklearn.autolog(log_input_examples=True, log_models=False, silent=True)

    # Infer model output schema
    try:
        output_schema = _infer_schema(y_train)
    except Exception as e:
        warnings.warn(f"Could not infer model output schema: {e}")
        output_schema = None

    # Infer model signature
    input_example = X_train[:3]  # Use a small subset as an example
    signature = infer_signature(X_train, best_model.predict(X_train))

    # Set model name for MLflow registration
    model_name = f"{catalog_name}.{schema_name}.hpo_model_ray_tune_optuna"

    # Display results
    print(f"Best Trial Number: {best_trial_number}")
    print(f"Best Hyperparameters: {best_model_params}")
    print(f"Best RMSE: {best_rmse:.4f}")

    # Log the best model to MLflow
    with mlflow.start_run(run_name="best_trial_ray_scikit_results", nested=True):
        mlflow.sklearn.log_model(
            sk_model=best_model,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
            registered_model_name=model_name,
            skops_trusted_types=["sklearn.tree._tree.Tree"]
        )

        mlflow.log_params(best_model_params)
        mlflow.log_metric("Best RMSE", best_rmse)

# Ensure MLflow run is properly closed
mlflow.end_run()

2026-09-19 02:46:00,179	WARNING optuna_search.py:346 -- You passed a `space` parameter to OptunaSearch that contained unresolved search space definitions. OptunaSearch should however be instantiated with fully configured search spaces only. To use Ray Tune's automatic search space conversion, pass the space definition as part of the `param_space` argument to `tune.Tuner()` instead.
[I 2026-09-19 02:46:00,180] A new study created in memory with name: optuna


+-----------------------------------------------------------------------------+
| Configuration for experiment     objective_ray_scikit_2026-09-19_02-46-00   |
+-----------------------------------------------------------------------------+
| Search algorithm                 SearchGenerator                            |
| Scheduler                        FIFOScheduler                              |
| Number of trials                 10                                         |
+-----------------------------------------------------------------------------+

View detailed results here: /home/spark-6c088b25-5785-4fcf-9f56-da/ray_results/objective_ray_scikit_2026-09-19_02-46-00
To visualize your results with TensorBoard, run: `tensorboard --logdir /local_disk0/tmp/ray/session_2026-09-19_02-40-21_463336_75/artifacts/2026-09-19_02-46-00/objective_ray_scikit_2026-09-19_02-46-00/driver_artifacts`

Trial status: 3 PENDING
Current time: 2026-09-19 02:46:00. Total running time: 0s
Logical resource 

(pid=1033) /databricks/python_shell/lib/third_party/python/vendor/protobuf/google/protobuf/internal/api_implementation.py:120: UserWarning: Selected implementation upb is not available. Falling back to the python implementation.
(pid=1033)   warnings.warn('Selected implementation upb is not available. '



Trial objective_ray_scikit_b9a98ed5 started with configuration:
+----------------------------------------------------+
| Trial objective_ray_scikit_b9a98ed5 config         |
+----------------------------------------------------+
| max_depth                                       23 |
| n_estimators                                   269 |
+----------------------------------------------------+

Trial objective_ray_scikit_cbe9a555 started with configuration:
+---------------------------------------------------+
| Trial objective_ray_scikit_cbe9a555 config        |
+---------------------------------------------------+
| max_depth                                      16 |
| n_estimators                                   85 |
+---------------------------------------------------+

Trial objective_ray_scikit_486995c7 started with configuration:
+----------------------------------------------------+
| Trial objective_ray_scikit_486995c7 config         |
+----------------------------------------

2026-09-19 02:46:24,097	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/spark-6c088b25-5785-4fcf-9f56-da/ray_results/objective_ray_scikit_2026-09-19_02-46-00' in 0.0074s.


...

/local_disk0/.ephemeral_nfs/envs/pythonEnv-6c088b25-5785-4fcf-9f56-da2e5cf63b34/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/09/19 02:46:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best Trial Number: 538ca153
Best Hyperparameters: {'n_estimators': 162, 'max_depth': 21, 'random_state': 42}
Best RMSE: 0.5872


🔗 View Logged Model at: https://dbc-0b69ef93-5247.cloud.databricks.com/ml/experiments/853238798589297/models/m-68b4cccbe8f5440bb1e4903818bb5628?o=7474657872577658
2026/09/19 02:46:38 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
Successfully registered model 'workspace.default.hpo_model_ray_tune_optuna'.


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.hpo_model_ray_tune_optuna': https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/models/workspace/default/hpo_model_ray_tune_optuna/version/1?o=7474657872577658


In [0]:
# página 20, clicar em experimento

In [0]:
#Aula do arquivo 6_2_MachineLearningatScale_depois...4.odt

Shut down the ray cluster.

In [0]:
ray.shutdown()

## Conclusion

In this demo, we explored how to setup and execute model training with Ray Tune on a single-machine. Additionally, we walked through the core components needed to perform model training with the Ray framework such as defining a search space, building objective functions, and optimization of hyperparameters.

In [0]:
# Start the parent MLflow run
#with mlflow.start_run(run_name="ray_tune", experiment_id=experiment_id_ray) as parent_run:
#    os.environ.update(mlflow_db_creds)  # Ensure Ray executors have credentials[cite: 15]

#    # Set up and execute Ray Tune with the objective function and search space[cite: 15]
#    tuner = tune.Tuner(
#        tune.with_parameters(
#            objective_ray_scikit,
#            parent_run_id=parent_run.info.run_id,
#            X_train_in=X_train,
#            y_train_in=y_train,
#            experiment_name_in=experiment_name_ray,
#            mlflow_db_creds_in=mlflow_db_creds
#        ),
#        tune_config=tune.TuneConfig(
#            search_alg=algo,
#            num_samples=10,
#            reuse_actors=True  # Keeps actors alive for efficiency[cite: 15]
#        ),
#        param_space=search_space
#    )


#    # Run tuning and retrieve the best result
#    multinode_results = tuner.fit()
#    best_result = multinode_results.get_best_result(metric="rmse", mode="min", scope="last")

#    if best_result is None:
#        raise ValueError("No best trial found. Ensure the tuning job ran successfully.")

    # Extract best trial details
#    best_trial_number = best_result.metrics.get("trial_id", "N/A")  # Default if missing
#    best_model_params = best_result.config
#    best_model_params["random_state"] = 42  # Ensures reproducibility
#    best_rmse = best_result.metrics["rmse"]

    # Train the best model using the best hyperparameters
    # Train the best model using the best hyperparameters
#    best_model = RandomForestRegressor(**best_model_params)
#    best_model.fit(X_train, y_train)

    # Enable MLflow autologging (disable model logging to avoid conflicts)
#    mlflow.sklearn.autolog(log_input_examples=True, log_models=False, silent=True)

    # Infer model output schema
#    try:
#        output_schema = _infer_schema(y_train)
#    except Exception as e:
#        warnings.warn(f"Could not infer model output schema: {e}")
#        output_schema = None

    # Infer model signature
#    input_example = X_train[:3]  # Use a small subset as an example
#    signature = infer_signature(X_train, best_model.predict(X_train))

    # Set model name for MLflow registration
#    model_name = f"{catalog_name}.{schema_name}.hpo_model_ray_tune_optuna"

    # Display results
#    print(f"Best Trial Number: {best_trial_number}")
#    print(f"Best Hyperparameters: {best_model_params}")
#    print(f"Best RMSE: {best_rmse:.4f}")

    # Log the best model to MLflow
#    with mlflow.start_run(run_name="best_trial_ray_scikit_results", nested=True):
#        mlflow.sklearn.log_model(
#            sk_model=best_model,
#            artifact_path="model",
#            signature=signature,
#            input_example=input_example,
#            registered_model_name=model_name
#        )

#        mlflow.log_params(best_model_params)
#        mlflow.log_metric("Best RMSE", best_rmse)

# Ensure MLflow run is properly closed
#mlflow.end_run()